# Data Exploration & Visualization
## PixLearn - Results Visualization Notebook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.datasets import mnist, cifar10

%matplotlib inline

### 1. Dataset Exploration

In [ ]:
(x_train_mnist, y_train_mnist), (x_test_mnist, y_test_mnist) = mnist.load_data()
(x_train_cifar, y_train_cifar), (x_test_cifar, y_test_cifar) = cifar10.load_data()

cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                 'dog', 'frog', 'horse', 'ship', 'truck']

print(f"MNIST: Train={x_train_mnist.shape}, Test={x_test_mnist.shape}")
print(f"CIFAR-10: Train={x_train_cifar.shape}, Test={x_test_cifar.shape}")

### 2. MNIST Sample Visualization

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(12, 8))
axes = axes.flatten()
for i in range(15):
    axes[i].imshow(x_train_mnist[i], cmap='gray')
    axes[i].set_title(f"Label: {y_train_mnist[i]}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()

### 3. CIFAR-10 Sample Visualization

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()
for i in range(10):
    axes[i].imshow(x_train_cifar[i])
    axes[i].set_title(cifar_classes[y_train_cifar[i][0]])
    axes[i].axis('off')
plt.tight_layout()
plt.show()

### 4. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(y_train_mnist, bins=10, edgecolor='black')
axes[0].set_title('MNIST Class Distribution')
axes[0].set_xlabel('Digit')
axes[0].set_ylabel('Count')

axes[1].hist(y_train_cifar, bins=10, edgecolor='black')
axes[1].set_title('CIFAR-10 Class Distribution')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].set_xticks(range(10))
axes[1].set_xticklabels([c[:4] for c in cifar_classes], rotation=45)

plt.tight_layout()
plt.show()

### 5. Model Predictions Visualization

In [ ]:
def visualize_predictions(model, x_test, y_test, dataset_name='mnist', num_samples=10):
    predictions = model.predict(x_test[:num_samples])
    predicted_labels = np.argmax(predictions, axis=1)
    true_labels = y_test[:num_samples].flatten()
    
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()
    
    for i in range(num_samples):
        is_gray = len(x_test[i].shape) < 3
        axes[i].imshow(x_test[i].squeeze(), cmap='gray' if is_gray else None)
        
        color = 'green' if predicted_labels[i] == true_labels[i] else 'red'
        if dataset_name == 'cifar10':
            title = f"True: {cifar_classes[true_labels[i]]}\nPred: {cifar_classes[predicted_labels[i]]}"
        else:
            title = f"True: {true_labels[i]}\nPred: {predicted_labels[i]}"
        axes[i].set_title(title, color=color)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

### 6. Confusion Matrix Visualization

In [ ]:
def plot_confusion_matrix(model, x_test, y_test, classes, dataset_name='dataset'):
    predictions = model.predict(x_test)
    predicted_labels = np.argmax(predictions, axis=1)
    true_labels = y_test.flatten()
    
    cm = confusion_matrix(true_labels, predicted_labels)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'{dataset_name} Confusion Matrix')
    plt.show()

### 7. Training History Comparison

In [ ]:
def compare_histories(histories, names):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    for history, name in zip(histories, names):
        axes[0].plot(history.history['accuracy'], label=f'{name} Train')
        axes[0].plot(history.history['val_accuracy'], label=f'{name} Val', linestyle='--')
    
    axes[0].set_title('Accuracy Comparison')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)
    
    for history, name in zip(histories, names):
        axes[1].plot(history.history['loss'], label=f'{name} Train')
        axes[1].plot(history.history['val_loss'], label=f'{name} Val', linestyle='--')
    
    axes[1].set_title('Loss Comparison')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

### 8. Feature Map Visualization

In [ ]:
def visualize_feature_maps(model, image, layer_idx=0):
    from tensorflow.keras.models import Model
    
    layer_output = model.layers[layer_idx].output
    activation_model = Model(inputs=model.input, outputs=layer_output)
    
    activations = activation_model.predict(np.expand_dims(image, 0))
    
    num_filters = activations.shape[-1]
    size = int(np.ceil(np.sqrt(num_filters)))
    
    fig, axes = plt.subplots(size, size, figsize=(10, 10))
    axes = axes.flatten()
    
    for i in range(num_filters):
        axes[i].imshow(activations[0, :, :, i], cmap='viridis')
        axes[i].axis('off')
    
    for i in range(num_filters, len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()